# A Fragile Equilibrium — guided walkthrough

This notebook calls the same modules as `src/report.py` and the manuscript build, so
anything you see here is what the paper reports. If they disagree, your kernel is stale.

**Before running:** populate `data/raw/` per `data/queries/cdc_wonder_queries.md`.
Cell 2 will raise a clear error if you have not.


In [ ]:
import sys, json
from pathlib import Path

# Make the repo root importable regardless of where Jupyter was launched
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import loader, rates, decomposition, excess, figures, report

pd.set_option('display.width', 120)
print('repo root:', ROOT)

## 1. Load and validate

`load_all` refuses incomplete or unverified data. An exception here is the loader
doing its job, not a bug.

In [ ]:
ds = loader.load_all(strict=True)
print('years:', ds.years)
print()
ds.annual_deaths.head()

## 2. Crude vs age-adjusted

The divergence between these two series is the entire subject of the paper.

In [ ]:
crude = rates.crude_rate(ds.annual_deaths, ds.population)
adjusted = rates.age_adjusted_rate(ds.by_age, ds.standard_pop)

comparison = crude.merge(adjusted, on='year')[['year','crude_rate','age_adjusted_rate']]
comparison['gap'] = comparison['crude_rate'] - comparison['age_adjusted_rate']
comparison.round(1)

A widening `gap` means age structure is contributing more and more to the crude rate.

In [ ]:
figures.fig_crude_vs_adjusted(crude, adjusted)
from IPython.display import Image
Image(str(ROOT / 'figures' / 'fig1_crude_vs_adjusted.png'))

## 3. Kitagawa decomposition

Splitting the crude-rate change into a mortality component and a demographic component.
The two sum to the total exactly.

In [ ]:
y0, y_pre, y_last = ds.years[0], 2019, ds.years[-1]

for a, b in [(y0, y_pre), (y_pre, y_last), (y0, y_last)]:
    r = decomposition.kitagawa(ds.by_age, a, b)
    print(r.summary())
    print(f'    age:rate ratio = {r.ratio:.2f}')
    resid = (r.rate_effect + r.age_effect) - r.total_change
    print(f'    additivity residual = {resid:.2e}')


The residual should be zero to floating-point precision. It is an algebraic identity.

## 4. Excess mortality

Baseline fitted on the age-adjusted rate over the pre-pandemic window, then projected.

In [ ]:
ex = excess.excess_mortality(
    adjusted, ds.by_age, ds.standard_pop,
    ds.annual_deaths[['year','deaths']], ds.population,
    baseline_start=y0, baseline_end=y_pre,
)

print(f'baseline {ex.baseline_years}, slope {ex.slope_per_year:+.3f} per year')
print(f'excess 2020-2021: {ex.total_excess(2020, 2021):,.0f}')

ex.table[['year','deaths','expected_deaths','excess_deaths','excess_pct']].round(1)

### Baseline sensitivity

The manuscript's limitations section requires reporting this. Run it here.

In [ ]:
for start in [y0, y0 + 2, y0 + 4]:
    alt = excess.excess_mortality(
        adjusted, ds.by_age, ds.standard_pop,
        ds.annual_deaths[['year','deaths']], ds.population,
        baseline_start=start, baseline_end=y_pre,
    )
    print(f'baseline {start}-{y_pre}: 2020-21 excess = {alt.total_excess(2020,2021):>12,.0f}')

## 5. Age distribution of COVID-19 deaths

In [ ]:
share = excess.covid_share_by_age(ds.covid_by_age)
share.round(1)

## 6. Build everything

Regenerates results.json, all five figures, and the manuscript.

In [ ]:
res = report.compute()
path = report.build_manuscript(res)
print('manuscript written to', path)

built = path.read_text()
assert '{{' not in built, 'unresolved tokens remain in the manuscript'
print('all template tokens resolved')